# DSRL offline-to-online 실행 노트북 v3 (Colab Pro+)

v2와 다른 점
- 코드를 포크(`msp0617/dsrl`, 브랜치 `o2o`)에서 받는다. 노트북에서 sed로 덧칠하던 패치가 코드에 들어갔다.
- 체크포인트/resume, CSV 로깅, 환경 스텝 단위 예산이 들어갔다. 세션이 끊기면 같은 명령을 다시 실행하면 이어진다.
- site-packages 패치는 `colab/patch_env.py` 한 번으로 끝난다.

설치 셀(1~8)은 v2에서 실제로 성공한 순서를 그대로 유지했다.
**세션 정책**: 코드/패키지는 매 세션 새로 설치, Drive에는 체크포인트·로그·config만 보관.

## 0. Drive 마운트

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJ = '/content/drive/MyDrive/dsrl_project'
for d in ['ckpt', 'logs', 'cfg_backup', 'dppo_log']:
    os.makedirs(f'{PROJ}/{d}', exist_ok=True)
print('project dir:', PROJ)

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. condacolab
아래 실행하면 **커널이 자동 재시작**됩니다. 정상이니 재시작 후 2번부터 이어서 실행하세요.
(이미 설치했다면 건너뛰기)

In [ ]:
!pip install -q condacolab
import condacolab
condacolab.install()   # 여기서 커널 재시작

## 2. 재시작 후: Drive 다시 마운트

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
PROJ = '/content/drive/MyDrive/dsrl_project'
print('ok')

## 3. 저장소 클론
`.gitmodules`가 ssh 주소라서 그냥 클론하면 서브모듈(dppo, stable-baselines3)이 빈 폴더가 됨.
아래 `insteadOf` 설정이 그걸 막아줌.

In [ ]:
%%bash
git config --global url."https://github.com/".insteadOf "git@github.com:"

cd /content
rm -rf dsrl
git clone --recurse-submodules -b o2o https://github.com/msp0617/dsrl.git
cd dsrl

git log --oneline -1
echo "=== dppo ==="
ls dppo | head -3
echo "=== stable-baselines3 ==="
ls stable-baselines3 | head -3

두 폴더에 파일이 보여야 함. 비어 있으면 아래 셀로 서브모듈만 다시 받기.

In [ ]:
%%bash
cd /content/dsrl
git submodule sync --recursive
git submodule update --init --recursive
ls dppo | head

## 4. conda 환경 (Python 3.10)

In [ ]:
%%bash
source /usr/local/etc/profile.d/conda.sh
conda env remove -n dsrl -y 2>/dev/null
conda create -n dsrl python=3.10 -y -q
echo created

## 5. 설치
5~15분 걸립니다. `-q`를 뺐으니 진행 상황이 보임.

In [ ]:
%%bash
source /usr/local/etc/profile.d/conda.sh
conda activate dsrl
cd /content/dsrl/dppo
pip install -e .
pip install -e ".[robomimic]"
echo "=== dppo done ==="


In [ ]:
%%bash
source /usr/local/etc/profile.d/conda.sh
conda activate dsrl
cd /content/dsrl/stable-baselines3
pip install -e .
pip install gdown
echo "=== sb3 done ==="


### 검증

In [ ]:
%%bash
set -e
source /usr/local/etc/profile.d/conda.sh
conda activate dsrl

python -m pip install "mujoco==3.1.6"

python -c "import mujoco; print('mujoco', mujoco.__version__)"

In [ ]:
%%bash
set -e
source /usr/local/etc/profile.d/conda.sh
conda activate dsrl

# v2에서 실제로 성공한 조합을 한 번에, 실패하는 단계 없이 설치한다.
# robomimic을 git master에서 받으면 egl_probe 빌드가 실패해 통째로 롤백되고
# torch 2.14까지 끌고 오므로, PyPI의 0.3.0을 쓴다.

echo "=== 1. robosuite v1.4.1 ==="
python -m pip install "cython<3" patchelf
python -m pip install \
  "robosuite @ git+https://github.com/ARISE-Initiative/robosuite.git@v1.4.1"

echo "=== 2. egl_probe (구버전 CMake, build isolation 없이) ==="
python -m pip install "cmake==3.31.6"
python -m pip install --no-build-isolation "egl_probe==1.0.2"

echo "=== 3. robomimic 0.3.0 ==="
python -m pip install "robomimic==0.3.0"

echo "=== 4. 버전 고정 (robosuite가 올려놓은 numpy 2.x 되돌리기; 마지막에 해야 안 뒤집힘) ==="
python -m pip install --force-reinstall "numpy==1.26.4" "opencv-python==4.9.0.80"
echo "=== 5. torch: G4(RTX PRO 6000 Blackwell, sm_120)는 CUDA 12.8+ 필요. 그 외 GPU는 검증된 2.4.0 유지 ==="
CC=$(nvidia-smi --query-gpu=compute_cap --format=csv,noheader | head -1)
echo "compute capability $CC"
if [ "${CC%%.*}" -ge 12 ]; then
  python -m pip install "torch==2.7.1" "torchvision==0.22.1" --index-url https://download.pytorch.org/whl/cu128
else
  python -m pip install "torch==2.4.0" "torchvision==0.19.0"
fi

echo "=== install done ==="

In [ ]:
%%bash
set -e
source /usr/local/etc/profile.d/conda.sh
conda activate dsrl

export MUJOCO_GL=egl
export PYOPENGL_PLATFORM=egl

python - <<'PY'
import sys
print("python     ", sys.executable)

import numpy
print("numpy      ", numpy.__version__)

import torch
print("torch      ", torch.__version__, "| cuda:", torch.cuda.is_available())
# 커널이 실제로 이 GPU에서 도는지 확인. Blackwell에 구형 torch를 올리면 여기서
# "no kernel image is available"로 죽는다.
print("gpu        ", torch.cuda.get_device_name(0), "| compute cap", torch.cuda.get_device_capability(0))
x = torch.randn(256, 256, device="cuda")
print("matmul ok  ", float((x @ x).sum()) == float((x @ x).sum()))

import torchvision
print("torchvision", torchvision.__version__)

import mujoco
print("mujoco     ", mujoco.__version__)

import robomimic
print("robomimic  ", robomimic.__version__)

import robosuite
print("robosuite  ", robosuite.__version__)

import stable_baselines3 as sb3
print("sb3        ", sb3.__version__)
PY

### 5b. conda 환경 캐시 (선택, 재설치 15분 → 3~5분)

런타임이 죽으면 VM 디스크가 사라져 4~5번을 매번 다시 한다. 설치가 끝난 환경을 Drive에 압축해 두면
다음 VM에서는 **1번(condacolab) → 2번(Drive) → 3번(클론) → 아래 "복원" 셀 → 6번부터** 로 간다.
4번, 5번은 건너뛴다. torch 2.7.1 cu128이 들어 있으므로 A100·G4 어느 쪽에서도 쓸 수 있다.
코드(`dppo`, `stable-baselines3`)는 editable 설치라 `/content/dsrl`에 클론이 있어야 한다.

In [ ]:
%%bash
# 저장: 설치가 끝난 뒤 한 번. 6~8GB, 5분 안팎.
CACHE=/content/drive/MyDrive/dsrl_project/env_cache
mkdir -p $CACHE
cd /usr/local/envs
tar -cf - dsrl | gzip -1 > $CACHE/dsrl_env.tar.gz.tmp && mv $CACHE/dsrl_env.tar.gz.tmp $CACHE/dsrl_env.tar.gz
ls -lh $CACHE/dsrl_env.tar.gz

In [ ]:
%%bash
# 복원: 새 VM에서 4~5번 대신. 1번(condacolab), 2번(Drive), 3번(클론) 뒤에 실행.
set -e
CACHE=/content/drive/MyDrive/dsrl_project/env_cache
mkdir -p /usr/local/envs
cd /usr/local/envs
rm -rf dsrl
tar -xzf $CACHE/dsrl_env.tar.gz
source /usr/local/etc/profile.d/conda.sh
conda activate dsrl
python - <<'PY'
import torch, robomimic, robosuite, mujoco, stable_baselines3
print("torch", torch.__version__, "| cuda", torch.cuda.is_available(), "| cap", torch.cuda.get_device_capability(0))
x = torch.randn(256, 256, device="cuda"); print("matmul ok", (x @ x).sum().item() != 0)
print("robomimic", robomimic.__version__, "robosuite", robosuite.__version__, "mujoco", mujoco.__version__)
PY
echo "restored; continue from section 6"

In [ ]:
%%bash
source /usr/local/etc/profile.d/conda.sh
conda activate dsrl
python /usr/local/envs/dsrl/lib/python3.10/site-packages/robosuite/scripts/setup_macros.py

## 6. π_dp 체크포인트
README 링크의 Drive 폴더를 `dppo/log`에 배치. 첫 세션만 다운로드하고 Drive에 복사해두면 다음부터는 복사만.

In [ ]:
%%bash
source /usr/local/etc/profile.d/conda.sh
conda activate dsrl
PROJ=/content/drive/MyDrive/dsrl_project
mkdir -p /content/dsrl/dppo/log

if [ -z "$(ls -A $PROJ/dppo_log 2>/dev/null)" ]; then
  echo "--- 첫 다운로드 ---"
  cd /content/dsrl/dppo/log
  gdown --folder https://drive.google.com/drive/folders/1kzC49RRFOE7aTnJh_7OvJ1K5XaDmtuh1
  cp -r /content/dsrl/dppo/log/. $PROJ/dppo_log/
else
  echo "--- Drive에서 복사 ---"
  cp -r $PROJ/dppo_log/. /content/dsrl/dppo/log/
fi
find /content/dsrl/dppo/log -maxdepth 3 | head -40

In [ ]:
%%bash
cd /content/dsrl

for FILE in \
  dppo/log/robomimic-pretrain/can/can_pre_diffusion_mlp_ta4_td20/2024-06-28_13-29-54/checkpoint/state_5000.pt \
  dppo/log/robomimic/can/normalization.npz
do
  test -f "$FILE" && echo "OK: $FILE" || echo "MISSING: $FILE"
done

In [ ]:
%%bash
set -e

RUNTIME=/content/dsrl/dppo/log
DRIVE=/content/drive/MyDrive/dsrl_project/dppo_log

CKPT_REL=robomimic-pretrain/can/can_pre_diffusion_mlp_ta4_td20/2024-06-28_13-29-54/checkpoint/state_5000.pt
NORM_REL=robomimic/can/normalization.npz

mkdir -p "$RUNTIME" "$DRIVE"

CKPT_SRC=$(find "$RUNTIME" "$DRIVE" \
  -type f -path "*/$CKPT_REL" -print -quit 2>/dev/null || true)

NORM_SRC=$(find "$RUNTIME" "$DRIVE" \
  -type f -path "*/$NORM_REL" -print -quit 2>/dev/null || true)

echo "checkpoint: ${CKPT_SRC:-NOT_FOUND}"
echo "normalization: ${NORM_SRC:-NOT_FOUND}"

if [[ -z "$CKPT_SRC" || -z "$NORM_SRC" ]]; then
  echo "기존 다운로드에서 파일을 찾지 못했습니다."
  exit 2
fi

CKPT_DST="$RUNTIME/$CKPT_REL"
NORM_DST="$RUNTIME/$NORM_REL"

mkdir -p "$(dirname "$CKPT_DST")" "$(dirname "$NORM_DST")"

[[ "$CKPT_SRC" == "$CKPT_DST" ]] || cp -f "$CKPT_SRC" "$CKPT_DST"
[[ "$NORM_SRC" == "$NORM_DST" ]] || cp -f "$NORM_SRC" "$NORM_DST"

# 다음 세션을 위해 Drive에도 정확한 구조로 저장
mkdir -p "$DRIVE/$(dirname "$CKPT_REL")"
mkdir -p "$DRIVE/$(dirname "$NORM_REL")"
cp -f "$CKPT_DST" "$DRIVE/$CKPT_REL"
cp -f "$NORM_DST" "$DRIVE/$NORM_REL"

echo "=== READY ==="
ls -lh "$CKPT_DST" "$NORM_DST"

## 7. 환경 변수
headless 렌더링과 wandb 비활성 설정.

In [ ]:
%%bash
cat > /content/env.sh <<'EOS'
export MUJOCO_GL=egl
export PYOPENGL_PLATFORM=egl
export WANDB_MODE=disabled
EOS

cat /content/env.sh

## 7b. 실행 헬퍼
긴 명령은 아래 `run_bash`로 돌린다. 출력이 실시간으로 보이고 로그가 Drive에 남는다.

In [ ]:
# Colab의 %%bash 는 명령이 끝나야 출력을 보여준다. 긴 학습은 이 헬퍼로 돌려서
# 한 줄씩 바로 보이게 하고, 같은 내용을 Drive의 로그 파일에도 남긴다.
import subprocess, sys

NOISE = ("Gym has been unmaintained", "Please upgrade to Gymnasium", "See the migration guide")

def run_bash(script, log_path=None):
    prefix = (
        "source /usr/local/etc/profile.d/conda.sh && conda activate dsrl\n"
        "source /content/env.sh\n"
        "cd /content/dsrl\n"
        "export HYDRA_FULL_ERROR=1 PYTHONUNBUFFERED=1\n"
    )
    log = open(log_path, "a") if log_path else None
    p = subprocess.Popen(["bash", "-c", prefix + script], stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        if log:
            log.write(line); log.flush()
        if not line.startswith(NOISE):
            print(line, end="", flush=True)
    p.wait()
    if log:
        log.close()
    print(f"\n[exit {p.returncode}]")
    return p.returncode

PROJ = "/content/drive/MyDrive/dsrl_project"
print("run_bash ready")

## 8. Config 확인
Can config의 키 이름을 보고 smoke test용 override를 정합니다 (총 step 수, 저장 주기 등).

In [ ]:
!cat /content/dsrl/cfg/robomimic/dsrl_can.yaml

## 9. site-packages 패치
robomimic이 mujoco_py를 무조건 import하는 문제를 고친다. **세션마다 한 번** 실행.

In [ ]:
%%bash
source /usr/local/etc/profile.d/conda.sh
conda activate dsrl
python /content/dsrl/colab/patch_env.py

## 10. 스모크 테스트 + resume 확인
작은 설정으로 두 번 나눠 돌린다. 두 번째 실행이 `[resume]` 을 찍고 이어가면 정상.

In [ ]:
run_bash(r'''
PROJ=/content/drive/MyDrive/dsrl_project
python train_dsrl.py --config-path=cfg/robomimic --config-name=dsrl_can.yaml \
  exp_id=smoke_resume \
  log_dir=$PROJ/logs \
  env.n_envs=1 env.n_eval_envs=1 num_evals=1 \
  eval_schedule.every_env_early=400 eval_schedule.early_until_env=100000 eval_schedule.num_evals_early=1 \
  ckpt_every_env_steps=400 \
  train.init_rollout_steps=50 train.utd=1 train.noise_critic_grad_steps=1 \
  train.batch_size=32 train.layer_size=256 train.num_layers=2 \
  train.buffer_size=20000 train.total_env_steps=1200
''', log_path=f"{PROJ}/logs/smoke_resume.out")

In [ ]:
run_bash(r'''
PROJ=/content/drive/MyDrive/dsrl_project
# 같은 명령에 목표만 늘린다. 앞 실행의 체크포인트에서 이어져야 한다.
python train_dsrl.py --config-path=cfg/robomimic --config-name=dsrl_can.yaml \
  exp_id=smoke_resume \
  log_dir=$PROJ/logs \
  env.n_envs=1 env.n_eval_envs=1 num_evals=1 \
  eval_schedule.every_env_early=400 eval_schedule.early_until_env=100000 eval_schedule.num_evals_early=1 \
  ckpt_every_env_steps=400 \
  train.init_rollout_steps=50 train.utd=1 train.noise_critic_grad_steps=1 \
  train.batch_size=32 train.layer_size=256 train.num_layers=2 \
  train.buffer_size=20000 train.total_env_steps=2000
''', log_path=f"{PROJ}/logs/smoke_resume.out")

In [ ]:
PROJ = '/content/drive/MyDrive/dsrl_project'
!ls -lh {PROJ}/logs/smoke_resume {PROJ}/logs/smoke_resume/checkpoint
!cat {PROJ}/logs/smoke_resume/checkpoint/run_state.json
!head -3 {PROJ}/logs/smoke_resume/eval_log.csv

## 11. 처리량 측정
**본 실험 하이퍼파라미터 그대로**, 초기 rollout만 줄여서 학습 구간 속도를 잰다.
결과의 `env steps/s` 로 300k 한 run에 몇 시간 걸리는지 나온다.

In [ ]:
run_bash(r'''
PROJ=/content/drive/MyDrive/dsrl_project
# init rollout 200스텝(=3200 env step) + 학습 10000 env step
time python train_dsrl.py --config-path=cfg/robomimic --config-name=dsrl_can.yaml \
  exp_id=tput_can seed=0 resume=False \
  log_dir=$PROJ/logs \
  train.init_rollout_steps=200 train.total_env_steps=13200 \
  eval_schedule.every_env_early=5000 \
  ckpt_every_env_steps=5000 save_replay_buffer=False
''', log_path=f"{PROJ}/logs/tput_can.out")

In [ ]:
!source /usr/local/etc/profile.d/conda.sh && conda activate dsrl && \
 python /content/dsrl/colab/throughput.py \
   /content/drive/MyDrive/dsrl_project/logs/tput_can --target 300000

## 12a. 오프라인 데이터 변환 + 사전학습 (warmup / iql 변형용)
시뮬레이터를 안 쓰므로 A100이 필요 없다. T4 세션에서 돌려도 된다.
- 변환은 한 번만 하면 된다 (Drive에 저장).
- 사전학습 결과 `.pt`는 시드별로 하나씩, 3개 온라인 run이 재사용한다.

In [ ]:
run_bash(r'''
PROJ=/content/drive/MyDrive/dsrl_project
CKPT=$PROJ/dppo_log/dsrl_public_checkpoints/robomimic/can
RAW=$PROJ/robomimic_raw/can/mh/low_dim_v141.hdf5
DST=$PROJ/offline/can_train_offline.npz
mkdir -p $PROJ/offline $PROJ/robomimic_raw

# 1) robomimic can-mh 원본 (보상 포함). 공개된 train.npz에는 rewards가 없다. 한 번만.
if [ ! -f $RAW ]; then
  python -m robomimic.scripts.download_datasets --tasks can --dataset_types mh \
    --hdf5_types low_dim --download_dir $PROJ/robomimic_raw
fi
ls -lh $RAW

# 2) 공개 normalization.npz로 정규화해 청크로 재구성. --check_against 가 공개 train.npz와
#    상태가 완전히 일치하는지 검사하므로, 통과하면 온라인과 같은 입력 공간이라는 증거다.
python scripts/make_offline_chunks.py \
  --load_path $RAW \
  --normalization_path $CKPT/normalization.npz \
  --check_against $CKPT/train.npz \
  --save_path $DST --n_envs 4
''')

In [ ]:
run_bash(r'''
PROJ=/content/drive/MyDrive/dsrl_project

METHOD=iql      # iql | warmup
SEED=1

python offline_pretrain.py --config-path=cfg/robomimic --config-name=dsrl_can.yaml \
  pretrain.method=$METHOD seed=$SEED \
  offline_data_path=$PROJ/offline/can_train_offline.npz \
  log_dir=$PROJ/logs
''', log_path=f"{PROJ}/logs/pretrain.out")

## 12b. 본 실험
한 세션에 한 run. `VARIANT`/`SEED`만 바꿔서 세션을 나눠 띄운다.
백그라운드로 돌리므로 셀은 바로 끝나고, 13번으로 진행을 본다.
세션이 죽으면 **이 셀을 그대로 다시 실행**하면 이어진다.
`warmup`/`iql`은 12a에서 같은 시드로 만든 `.pt`가 있어야 한다.

In [ ]:
%%bash
source /usr/local/etc/profile.d/conda.sh && conda activate dsrl
source /content/env.sh
cd /content/dsrl

VARIANT=baseline      # baseline | warmup | iql
SEED=1
MIX=none              # none | prefill | fixed | linear   (offline replay 비율 축, 200k)
STEPS=""              # 비우면 config 기본(300k; MIX!=none이면 200k). 예: 150000
EXTRA_ARGS=""         # 추가 override. 예: "train.ent_coef=0.01"  (이때는 EXP_TAG도 줄 것)
EXP_TAG=""            # exp_id 뒤에 붙는 꼬리표. 예: fixalpha -> can_fixalpha_s1
PROJ=/content/drive/MyDrive/dsrl_project

EXTRA=""
EXP=can_${VARIANT}_s${SEED}
if [ "$VARIANT" != "baseline" ]; then
  EXTRA="pretrain_path=$PROJ/logs/pretrain/${VARIANT}_can_s${SEED}.pt"
fi
case "$MIX" in
  none) ;;
  prefill) EXTRA="$EXTRA offline_mix.mode=prefill" ;;
  fixed)   EXTRA="$EXTRA offline_mix.mode=fixed offline_mix.p0=0.5" ;;
  linear)  EXTRA="$EXTRA offline_mix.mode=linear offline_mix.p0=0.8 offline_mix.p1=0.1 offline_mix.until_env=100000" ;;
  *) echo "unknown MIX=$MIX"; exit 1 ;;
esac
if [ "$MIX" != "none" ]; then
  EXTRA="$EXTRA train.total_env_steps=200000 offline_data_path=$PROJ/offline/can_train_offline.npz"
  if [ "$VARIANT" = "baseline" ]; then EXP=can_mix_${MIX}_s${SEED}; else EXP=can_${VARIANT}_${MIX}_s${SEED}; fi
fi
if [ -n "$STEPS" ]; then EXTRA="$EXTRA train.total_env_steps=$STEPS"; fi
if [ -n "$EXTRA_ARGS" ]; then EXTRA="$EXTRA $EXTRA_ARGS"; fi
if [ -n "$EXP_TAG" ]; then
  if [ "$VARIANT" = "baseline" ]; then EXP=can_${EXP_TAG}_s${SEED}; else EXP=can_${VARIANT}${EXP_TAG}_s${SEED}; fi
fi

nohup python train_dsrl.py --config-path=cfg/robomimic --config-name=dsrl_can.yaml \
  exp_id=$EXP seed=$SEED variant=$VARIANT $EXTRA \
  log_dir=$PROJ/logs \
  > $PROJ/logs/${EXP}.out 2>&1 &

echo "started $EXP (pid $!) -> $PROJ/logs/${EXP}.out"


## 13. 진행 확인

In [ ]:
EXP = 'can_baseline_s1'
PROJ = '/content/drive/MyDrive/dsrl_project'

!tail -n 3 {PROJ}/logs/{EXP}.out
!echo '--- eval ---' && tail -n 5 {PROJ}/logs/{EXP}/eval_log.csv
!echo '--- ckpt ---' && cat {PROJ}/logs/{EXP}/checkpoint/run_state.json
!source /usr/local/etc/profile.d/conda.sh && conda activate dsrl && \
 python /content/dsrl/colab/throughput.py {PROJ}/logs/{EXP} --target 300000

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

EXP = 'can_baseline_s1'
PROJ = '/content/drive/MyDrive/dsrl_project'
df = pd.read_csv(f'{PROJ}/logs/{EXP}/eval_log.csv')
df = df[df.deterministic == 0]
plt.figure(figsize=(6, 3.5))
plt.plot(df.env_steps, df.success_rate, marker='o', ms=3)
plt.xlabel('environment steps'); plt.ylabel('success rate'); plt.title(EXP)
plt.grid(alpha=.3); plt.show()

## 14. 세션 종료 전 백업

In [ ]:
%%bash
source /usr/local/etc/profile.d/conda.sh
conda activate dsrl
PROJ=/content/drive/MyDrive/dsrl_project
cp -r /content/dsrl/cfg $PROJ/cfg_backup/
pip freeze > $PROJ/requirements_lock.txt
ls -la $PROJ
echo done